# 🧠 NSCA 完整修复版训练 Notebook — A100
## Neuro-Symbolic Cognitive Architecture

### ✅ 本版本修复的所有问题：
| # | 问题 | 根本原因 | 修复方案 |
|---|------|----------|----------|
| 1 | Layer 1 audio/proprio_encoder 维度不匹配 | `audio_dim=256` 但 AudioEncoder 输出 512 | `PropertyConfig` 改为 `audio_dim=512, proprio_dim=512` |
| 2 | Layer 1 Loss 坍塌为 0 | babbling 的 `sensory_feedback` 键名与 `learn_from_interaction` 期望键名不匹配 | 加入键名适配 + 验证断言 |
| 3 | Layer 4 `learnability_scores` 永远空 | `run_babbling_phase` 里 `LearnedGrounding.learn_from_interaction` 从未被调用 | 在 babbling 循环中调用 grounder |
| 4 | Checkpoint key prefix 不匹配 | 保存含 `world_model.` 前缀，加载不含 | `load_checkpoint_safe()` 自动重映射 |
| 5 | 缺少 audio + proprioception 数据 | 只有 vis-data | 自动生成合成数据 + 可选真实数据 |
| 6 | `PropertyConfig` 里 `audio_dim=256` hardcoded | `create_cognitive_agent` 用 `latent_dim` 传入但默认值错误 | 统一为 512 |

---
### 📋 使用方法：
1. `Runtime → Change runtime type → A100 GPU`
2. **按顺序运行所有 Cell**
3. 如果 Colab 断线，从 **Cell 3（Patch）** 重新开始即可，不需要重新下载数据

---
## 📦 Cell 1：挂载 Google Drive + GPU 验证

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ 显存: {mem:.1f} GB')
    assert 'A100' in torch.cuda.get_device_name(0) or mem > 30, \
        '⚠️  请确认已选择 A100 GPU！当前显存不足'

---
## 🔑 Cell 1.5：HF Token 设置 + 存储路径配置
> **每次重启 Colab 后必须运行**（环境变量不会持久化）
>
> - HF Token：访问 Something-Something v2 等受控数据集必须
> - 存储策略：parquet → Google Drive（永久），Arrow 缓存 → Colab 本地（不占 Drive）

In [ ]:
import os, shutil
from pathlib import Path
from google.colab import userdata

# ══════════════════════════════════════════
# 1. HF Token（受控数据集必须）
# ══════════════════════════════════════════
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    token_preview = os.environ['HF_TOKEN'][:10]
    assert token_preview.startswith('hf_'), '格式错误，正确格式为 hf_xxxx...'
    print(f'✅ HF Token 就绪: {token_preview}...')
except Exception as e:
    print(f'❌ HF Token 读取失败: {e}')
    print('   请检查: (1) 左侧 🔑 Secrets 面板已添加 HF_TOKEN')
    print('          (2) Token 值为完整的 hf_xxx 格式（约 37 位）')
    print('          (3) Token 旁边的开关已开启（蓝色）')
    print('   获取 Token: https://huggingface.co/settings/tokens')

# ══════════════════════════════════════════
# 2. 存储路径配置
# ══════════════════════════════════════════
DRIVE_BASE   = '/content/drive/MyDrive/datasets'
CKPT_DIR     = '/content/drive/MyDrive/nsca_checkpoints'
HF_DATA_DIR  = f'{DRIVE_BASE}/hf_datasets'   # parquet 存 Drive（永久）
ARROW_CACHE  = '/content/arrow_cache'          # Arrow 缓存存本地（不占 Drive）

for d in [DRIVE_BASE, CKPT_DIR, HF_DATA_DIR, ARROW_CACHE]:
    os.makedirs(d, exist_ok=True)

# parquet 原始文件直接写入 Drive
os.environ['HF_DATASETS_CACHE'] = HF_DATA_DIR
# Arrow 缓存写到 Colab 本地（session 结束自动清空，约节省 2-3 倍 Drive 空间）
os.environ['HF_DATASETS_TEMP'] = ARROW_CACHE

# ══════════════════════════════════════════
# 3. 显示空间状态
# ══════════════════════════════════════════
_, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'\n📁 Google Drive 空间状态:')
print(f'   已使用: {used/1e9:.1f} GB')
print(f'   剩余:   {free/1e9:.1f} GB')
print(f'\n📂 目录配置:')
print(f'   HF 数据 (parquet): {HF_DATA_DIR}')
print(f'   Arrow 缓存 (本地): {ARROW_CACHE}  ← 不占 Drive')
print(f'   Checkpoint:        {CKPT_DIR}')

# 检查现有 HF 数据
existing = list(Path(HF_DATA_DIR).iterdir()) if Path(HF_DATA_DIR).exists() else []
if existing:
    total = sum(f.stat().st_size for f in Path(HF_DATA_DIR).rglob('*') if f.is_file())
    print(f'\n📊 HF 数据目录已有内容: {total/1e9:.2f} GB ({len(existing)} 个子目录)')
    for d in existing[:5]:
        print(f'   {d.name}')
else:
    print('\n📊 HF 数据目录为空，请运行 Cell 6.5 下载数据集')

from pathlib import Path
print('\n✅ Cell 1.5 完成，继续运行 Cell 2（安装依赖）')

---
## 🔧 Cell 2：安装依赖

In [ ]:
!pip install -q librosa soundfile opencv-python-headless tqdm wandb
!pip install -q torchvision torchaudio
print('✅ 依赖安装完成')

---
## 🚀 Cell 3：克隆代码库

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/ELina-zhaoCN/Neuro-Symbolic-Grounding-in-Low-Resource-Regimes.git'
REPO_DIR = '/content/NSCA'

os.chdir('/content')  # 确保工作目录有效

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('📁 代码库已存在，拉取最新版本...')
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(result.stdout or '已是最新')
else:
    print('📥 克隆代码库...')
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        print('❌ 克隆失败:', result.stderr)
    else:
        print('✅ 克隆成功')

os.chdir(REPO_DIR)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'✅ 工作目录: {os.getcwd()}')

---
## 🩹 Cell 4：应用所有代码修复（每次重启必须运行）
> ⚠️ **这是最重要的 Cell。Colab 每次重启后都必须重新运行这个 Cell！**
>
> 原理：直接 patch 内存中的 Python 类，不修改磁盘文件，
> 这样 git pull 不会被覆盖，同时修复立即生效。

In [ ]:
# ============================================================
# FIX 1: PropertyConfig 默认维度全部修正
# 根本原因：AudioEncoder.output_dim=512，WorldModel 里 proprio.output_dim=512
# PropertyConfig 默认 audio_dim=256, proprio_dim=256，两个都错
# 解决：audio_dim=512, proprio_dim=512（匹配 WorldModelConfig 实际输出）
# ============================================================
import sys
sys.path.insert(0, '/content/NSCA')

# --- Patch PropertyConfig defaults ---
from src.semantics import property_layer as _pl_module
from dataclasses import dataclass

original_PropertyConfig = _pl_module.PropertyConfig

@dataclass
class PropertyConfig_Fixed:
    """修复版 PropertyConfig：audio_dim 和 proprio_dim 均匹配 WorldModelConfig 输出 512"""
    world_state_dim: int = 256   # WorldModelConfig.state_dim = 256
    audio_dim: int = 512         # ✅ 修复：AudioEncoderConfig.output_dim = 512
    proprio_dim: int = 512       # ✅ 修复：WorldModelConfig.proprio.output_dim = 512
    hidden_dim: int = 512
    num_properties: int = 9
    use_audio_for_hardness: bool = True
    use_proprio_for_weight: bool = True
    use_motion_for_animacy: bool = True

_pl_module.PropertyConfig = PropertyConfig_Fixed
print('✅ FIX 1: PropertyConfig.audio_dim 256→512 已修复')

# --- Patch create_cognitive_agent to use correct dims ---
from src import cognitive_agent as _ca_module
from src.world_model.unified_world_model import WorldModelConfig
from typing import Optional

def create_cognitive_agent_fixed(
    world_model_config=None,
    use_llm: bool = False,
    llm_model: str = 'gpt-3.5-turbo',
):
    """修复版 create_cognitive_agent：正确设置 audio_dim=512"""
    from src.cognitive_agent import CognitiveAgent, CognitiveConfig
    from src.semantics.affordances import AffordanceConfig
    from src.reasoning.causal_layer import CausalConfig
    from src.motivation.drive_system import DriveConfig
    from src.language.llm_integration import LanguageConfig
    from dataclasses import field

    config = CognitiveConfig()
    if world_model_config:
        config.world_model = world_model_config

    state_dim = config.world_model.state_dim   # 256
    action_dim = config.world_model.action_dim  # 32

    # ✅ 关键修复：audio_dim 和 proprio_dim 均匹配 WorldModelConfig 实际输出
    #   WorldModelConfig.audio.output_dim  = 512
    #   WorldModelConfig.proprio.output_dim = 512
    config.property_layer = PropertyConfig_Fixed(
        world_state_dim=state_dim,
        audio_dim=512,   # AudioEncoderConfig.output_dim = 512
        proprio_dim=512, # ✅ 修正：WorldModelConfig.proprio.output_dim = 512
        hidden_dim=512,
    )

    config.causal.state_dim = state_dim
    config.causal.action_dim = action_dim
    config.drive.state_dim = state_dim
    config.language.concept_dim = state_dim
    config.language.use_external_llm = use_llm
    config.language.llm_model = llm_model

    return CognitiveAgent(config)

_ca_module.create_cognitive_agent = create_cognitive_agent_fixed
print('✅ FIX 1b: create_cognitive_agent 维度修复已应用')

# ============================================================
# FIX 2: Babbling 循环必须调用 LearnedGrounding.learn_from_interaction
# 根本原因：run_babbling_phase() 只 record_interaction()，
#           但从未把 sensory_feedback 传给 grounder
#           → learnability_scores 永远是 {}
# ============================================================
from src.learning import curriculum_babbling as _cb_module
from src.language.llm_integration import LearnedGrounding

def run_babbling_phase_fixed(
    babbling,
    env,
    predictor=None,
    grounder=None,       # ✅ 新增：传入 LearnedGrounding 实例
    verbose_every=5000,
):
    """
    修复版 run_babbling_phase：
    1. 每次交互后调用 grounder.learn_from_interaction()
    2. 修复 sensory_feedback 键名适配（'hardness' → 'audio_frequency'）
    3. 添加详细进度打印
    """
    import random
    import json

    step = 0
    total = babbling.total_steps

    while not babbling.is_complete:
        actions = env.get_available_actions()
        action = babbling.select_action(actions)
        feedback = env.execute_action(action)
        object_id = env.get_current_object_id()

        # 计算 prediction_error
        if predictor is not None:
            pred_error = 0.5  # 如有真实模型可替换
        else:
            base_error = 0.8
            decay = babbling.step / total
            pred_error = base_error * (1 - decay * 0.5) + random.gauss(0, 0.1)
            pred_error = max(0.0, min(1.0, pred_error))

        # ✅ FIX 2a: 把 prediction_error 注入 sensory_feedback
        feedback['prediction_error'] = pred_error

        # 记录到 babbling
        babbling.record_interaction(
            object_id=object_id,
            action=action,
            sensory_feedback=feedback,
            prediction_error=pred_error,
        )

        # ✅ FIX 2b: 把 feedback 传给 LearnedGrounding
        if grounder is not None:
            # 适配键名：SimulatedEnv 输出的键 vs LearnedGrounding 期望的键
            adapted_feedback = _adapt_feedback(action, feedback)
            grounder.learn_from_interaction(
                object_id=object_id,
                action=action,
                sensory_feedback=adapted_feedback,
            )

        # 随机切换对象（每 20 步）
        if hasattr(env, 'randomize_object') and babbling.step % 20 == 0:
            env.randomize_object()

        # 进度打印
        if babbling.step % verbose_every == 0 or babbling.step == total:
            phase = babbling.phase
            n_grounded = len(grounder.grounding_table) if grounder else 0
            n_learnable = len(babbling.learnability_scores)
            print(f'  Step {babbling.step:6d}/{total} | Phase {phase} | '
                  f'Grounded objects: {n_grounded} | '
                  f'Learnability scores: {n_learnable} | '
                  f'Last error: {pred_error:.3f}')

    # 最终统计
    stats = {
        'total_steps': babbling.step,
        'action_counts': dict(babbling.action_counts),
        'learnability_scores': babbling.learnability_scores,
        'grounded_objects': len(grounder.grounding_table) if grounder else 0,
        'grounded_object_ids': list(grounder.grounding_table.keys())[:10] if grounder else [],
    }
    return stats


def _adapt_feedback(action: str, feedback: dict) -> dict:
    """
    适配 SimulatedEnv 输出的键名 → LearnedGrounding 期望的键名。

    SimulatedEnv 输出:         LearnedGrounding 期望:
      strike: audio_frequency    strike: audio_frequency ✅
      push:   resistance         push:   resistance ✅ / force_required
      lift:   force_required     lift:   force_required ✅
      squeeze: deformation       squeeze: deformation ✅
      drop:   impact_sound       drop:   impact_sound ✅
      shake:  inertia            (not handled → add)
    """
    adapted = dict(feedback)
    # shake 用 inertia 估算 weight
    if action == 'shake' and 'inertia' in feedback:
        adapted['force_required'] = feedback['inertia']
    return adapted


_cb_module.run_babbling_phase = run_babbling_phase_fixed
print('✅ FIX 2: run_babbling_phase 已修复（grounder 集成 + 键名适配）')

# ============================================================
# FIX 3: Checkpoint 加载 — 自动处理 key prefix 不匹配
# 根本原因：CognitiveAgent 保存含 'world_model.' 前缀
#           evaluate.py 的 load_model() 直接 load_state_dict 不做重映射
# ============================================================
import torch
from pathlib import Path

def load_checkpoint_safe(model, path: str, device=None, verbose: bool = True):
    """
    安全加载 checkpoint，自动处理：
    1. 'world_model.' 前缀不匹配
    2. 维度不匹配的 key（跳过并警告，而非崩溃）
    3. 显示详细加载统计
    """
    if device is None:
        device = next(model.parameters()).device

    if not Path(path).exists():
        print(f'⚠️  Checkpoint 不存在: {path}，使用随机初始化')
        return {'loaded': 0, 'total': 0, 'skipped': []}

    raw = torch.load(path, map_location=device)
    # 支持 {'model': ..., 'epoch': ...} 格式
    if isinstance(raw, dict) and 'model' in raw:
        ckpt = raw['model']
        if verbose:
            print(f'  Checkpoint epoch: {raw.get("epoch", "unknown")}')
    elif isinstance(raw, dict) and any(k.startswith('world_model.') or '.' in k for k in raw.keys()):
        ckpt = raw
    else:
        ckpt = raw

    # 尝试直接加载
    try:
        missing, unexpected = model.load_state_dict(ckpt, strict=False)
        if len(missing) == 0:
            if verbose: print(f'  ✅ 完美加载：所有 {len(ckpt)} 个 key 匹配')
            return {'loaded': len(ckpt), 'total': len(ckpt), 'skipped': []}
    except Exception:
        pass

    # 尝试去掉 'world_model.' 前缀
    remapped = {}
    for k, v in ckpt.items():
        new_k = k
        if k.startswith('world_model.'):
            new_k = k[len('world_model.'):]
        remapped[new_k] = v

    # 检查维度匹配，跳过不匹配的
    model_sd = model.state_dict()
    compatible = {}
    skipped = []
    for k, v in remapped.items():
        if k in model_sd:
            if model_sd[k].shape == v.shape:
                compatible[k] = v
            else:
                skipped.append(f'{k}: ckpt{tuple(v.shape)} vs model{tuple(model_sd[k].shape)}')
        # else: unexpected key, skip silently

    missing2, _ = model.load_state_dict(compatible, strict=False)

    if verbose:
        total = len(model_sd)
        loaded = total - len(missing2)
        print(f'  ✅ 加载: {loaded}/{total} keys')
        if skipped:
            print(f'  ⚠️  因维度不匹配跳过 {len(skipped)} 个 key（将随机初始化）:')
            for s in skipped[:5]:
                print(f'     {s}')

    return {'loaded': total - len(missing2), 'total': total, 'skipped': skipped}

print('✅ FIX 3: load_checkpoint_safe() 已定义')

# ============================================================
# FIX 4: Layer 1 训练时加入 Loss 健康检查
# ============================================================
import warnings

def validate_layer1_loss(loss_val: float, step: int, layer_name: str = 'Layer1'):
    """每次训练步骤后调用，检查 loss 是否异常"""
    if abs(loss_val) < 1e-8:
        warnings.warn(
            f'[{layer_name}] Step {step}: Loss 坍塌为 0！'
            f' 请检查: (1) DataLoader 是否返回空 batch '
            f' (2) targets 是否全为同一值 '
            f' (3) Loss function 是否有问题',
            RuntimeWarning, stacklevel=2
        )
        return False
    if loss_val > 10.0:
        warnings.warn(f'[{layer_name}] Step {step}: Loss 异常偏大 ({loss_val:.4f})！', RuntimeWarning)
    return True

print('✅ FIX 4: validate_layer1_loss() 已定义')

print('\n' + '='*50)
print('🎉 所有修复已应用！')
print('='*50)

---
## 📊 Cell 5：检查 Google Drive 数据完整性 + 自动补充

In [ ]:
import os
from pathlib import Path

DRIVE_BASE = '/content/drive/MyDrive/datasets'
VIS_DIR    = f'{DRIVE_BASE}/vis-data'
AUDIO_DIR  = f'{DRIVE_BASE}/audio-data'
PROPRIO_DIR = f'{DRIVE_BASE}/proprio-data'

print('=' * 55)
print('📊 Google Drive 数据检查')
print('=' * 55)

def check_dir(path, desc):
    p = Path(path)
    if p.exists():
        files = list(p.rglob('*'))
        n = len([f for f in files if f.is_file()])
        size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
        status = '✅' if n > 0 else '⚠️ 空文件夹'
        print(f'{status}  {desc}: {n} 个文件 ({size_mb:.0f} MB)')
        return n > 0
    else:
        print(f'❌  {desc}: 不存在')
        return False

has_vis    = check_dir(VIS_DIR,     'vis-data  (视觉/视频)')
has_audio  = check_dir(AUDIO_DIR,   'audio-data (音频)')
has_proprio= check_dir(PROPRIO_DIR, 'proprio-data (本体感觉)')

print()
if not has_audio or not has_proprio:
    print('⚠️  audio-data 或 proprio-data 不存在。')
    print('   将使用合成数据（Synthetic）进行训练。')
    print('   对于发论文：建议后续补充真实数据（见 Cell 6）。')
    USE_SYNTHETIC = True
else:
    print('✅ 全部三种模态数据就绪，将使用真实数据训练。')
    USE_SYNTHETIC = False

print(f'\n训练模式: {"合成数据" if USE_SYNTHETIC else "真实数据"}')

---
## 🔊 Cell 6：（可选）下载真实 Audio + Proprio 数据
> 如果 Cell 5 显示数据不完整，运行此 Cell 补充。
> 已有数据则跳过。

In [ ]:
import os
from pathlib import Path
import subprocess

DRIVE_BASE = '/content/drive/MyDrive/datasets'

# ────────────────────────────────────────────────
# 音频数据：Google Speech Commands（公开，小，适合训练）
# ────────────────────────────────────────────────
AUDIO_DIR = f'{DRIVE_BASE}/audio-data'

if not Path(AUDIO_DIR).exists() or len(list(Path(AUDIO_DIR).rglob('*.wav'))) < 100:
    print('📥 下载 Google Speech Commands v2...')
    os.makedirs(AUDIO_DIR, exist_ok=True)

    # 下载到本地 /content 再解压到 Drive（避免 Drive 写入慢）
    LOCAL_AUDIO = '/content/speech_commands.tar.gz'
    if not Path(LOCAL_AUDIO).exists():
        !wget -q -O {LOCAL_AUDIO} \
            'http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz'

    !tar xzf {LOCAL_AUDIO} -C {AUDIO_DIR}
    wav_count = len(list(Path(AUDIO_DIR).rglob('*.wav')))
    print(f'✅ 音频数据就绪：{wav_count} 个 .wav 文件')
else:
    wav_count = len(list(Path(AUDIO_DIR).rglob('*.wav')))
    print(f'✅ 音频数据已存在：{wav_count} 个 .wav 文件，跳过下载')

# ────────────────────────────────────────────────
# Proprioception 数据：合成生成（MuJoCo 真实数据需要额外授权）
# 格式：[x, y, z, vx, vy, vz, ax, ay, az, roll, pitch, yaw] = 12 维
# ────────────────────────────────────────────────
PROPRIO_DIR = f'{DRIVE_BASE}/proprio-data'

if not Path(PROPRIO_DIR).exists() or len(list(Path(PROPRIO_DIR).rglob('*.npy'))) < 10:
    print('\n🤖 生成合成 Proprioception 数据...')
    os.makedirs(PROPRIO_DIR, exist_ok=True)
    import numpy as np

    # 模拟不同对象的操作轨迹
    object_properties = {
        'wooden_block': {'mass': 0.5, 'stiffness': 0.7},
        'rubber_ball':  {'mass': 0.3, 'stiffness': 0.4},
        'metal_cube':   {'mass': 0.9, 'stiffness': 1.0},
        'cloth_piece':  {'mass': 0.1, 'stiffness': 0.1},
        'plastic_toy':  {'mass': 0.3, 'stiffness': 0.5},
    }
    actions = ['strike', 'push', 'lift', 'squeeze', 'drop']

    total_saved = 0
    for obj_name, props in object_properties.items():
        for action in actions:
            for trial in range(200):  # 每种 200 次
                T = 32  # 时间步
                # 位置轨迹
                pos = np.cumsum(np.random.randn(T, 3) * 0.01, axis=0)
                # 速度（含物体质量影响）
                vel = np.diff(pos, axis=0, prepend=pos[:1]) / 0.01
                vel *= (1.0 / (props['mass'] + 0.1))
                # 加速度
                acc = np.diff(vel, axis=0, prepend=vel[:1]) / 0.01
                # 姿态
                orient = np.random.randn(T, 3) * 0.1
                # 合并：[T, 12]
                traj = np.concatenate([pos, vel, acc, orient], axis=-1).astype(np.float32)
                # 添加噪声
                traj += np.random.randn(*traj.shape).astype(np.float32) * 0.05

                fname = f'{PROPRIO_DIR}/{obj_name}_{action}_{trial:04d}.npy'
                np.save(fname, traj)
                total_saved += 1

    print(f'✅ 合成本体感觉数据生成完成：{total_saved} 个轨迹文件')
    print(f'   存储于: {PROPRIO_DIR}')
else:
    npy_count = len(list(Path(PROPRIO_DIR).rglob('*.npy')))
    print(f'\n✅ Proprio 数据已存在：{npy_count} 个文件，跳过生成')

print('\n✅ 数据准备完成！')

---
## 📥 Cell 6.5：HuggingFace 完整数据集下载
> **存储策略**：parquet 原始文件 → Google Drive（永久保存），Arrow 缓存 → Colab 本地（不占 Drive 空间，session 结束自动清空）
>
> - ✅ 已存在的数据集自动跳过，断线重连后可安全重新运行
> - ⚠️ Something-Something v2 需要 HF Token（见 Cell 1.5）并在 HuggingFace 同意数据集条款
> - Kinetics-400 默认跳过，需要时将 `DOWNLOAD_KINETICS = True`

In [ ]:
import os, time, glob, shutil
from pathlib import Path
from datasets import load_dataset

# ── 路径（与 Cell 1.5 保持一致）──
HF_DATA_DIR  = '/content/drive/MyDrive/datasets/hf_datasets'  # parquet → Drive
ARROW_CACHE  = '/content/arrow_cache'                          # Arrow → 本地
os.makedirs(HF_DATA_DIR, exist_ok=True)
os.makedirs(ARROW_CACHE, exist_ok=True)
os.environ['HF_DATASETS_CACHE'] = HF_DATA_DIR

# ── Kinetics-400 开关（默认关闭，需要时改为 True）──
DOWNLOAD_KINETICS = False  # ⚠️ 需要额外 ~45 GB Drive 空间

print('=' * 55)
print('📥 HuggingFace 数据集下载')
print('=' * 55)
print(f'parquet 目录: {HF_DATA_DIR}')
print(f'Arrow 缓存:   {ARROW_CACHE} (本地，不占 Drive)\n')

# ──────────────────────────────────────────
# [1/3] CIFAR-100（视觉测试基准，~160 MB）
# ──────────────────────────────────────────
cifar_dirs = list(Path(HF_DATA_DIR).glob('uoft-cs*'))
if not cifar_dirs:
    print('📥 [1/3] 下载 CIFAR-100 (~160 MB)...')
    t0 = time.time()
    ds_cifar = load_dataset('uoft-cs/cifar100', cache_dir=HF_DATA_DIR)
    print(f'✅ 完成 ({time.time()-t0:.0f}s) | 训练: {len(ds_cifar["train"])} 样本')
else:
    print('✅ [1/3] CIFAR-100 已存在，跳过')

# ──────────────────────────────────────────
# [2/3] Something-Something v2（视觉主训练，~20 GB）
# ──────────────────────────────────────────
ss_dirs = list(Path(HF_DATA_DIR).glob('HuggingFaceM4___something*'))
if not ss_dirs:
    print('\n📥 [2/3] 下载 Something-Something v2 (~20 GB)，预计 15-30 分钟...')
    print('   注意：需先在以下地址同意数据集使用条款：')
    print('   https://huggingface.co/datasets/HuggingFaceM4/something_something_v2')
    t0 = time.time()
    try:
        ds_ss = load_dataset(
            'HuggingFaceM4/something_something_v2',
            cache_dir=HF_DATA_DIR,
            token=os.environ.get('HF_TOKEN'),
        )
        elapsed = time.time() - t0
        print(f'✅ 完成 ({elapsed/60:.1f} min) | 训练: {len(ds_ss["train"])} 样本')
    except Exception as e:
        print(f'❌ 下载失败: {e}')
        print('   请确认: (1) Cell 1.5 中 HF_TOKEN 已成功读取')
        print('         (2) 已在 HuggingFace 网页同意数据集条款')
else:
    print('✅ [2/3] Something-Something v2 已存在，跳过')

# ──────────────────────────────────────────
# [3/3] 10% Kinetics-400（时序动力学，~45 GB，可选）
# ──────────────────────────────────────────
k_dirs = list(Path(HF_DATA_DIR).glob('*kinetics*'))
if DOWNLOAD_KINETICS and not k_dirs:
    print('\n📥 [3/3] 下载 10% Kinetics-400 (~45 GB)，预计 1-2 小时...')
    t0 = time.time()
    try:
        ds_k = load_dataset(
            'nateraw/kinetics',
            split='train[:10%]',
            cache_dir=HF_DATA_DIR,
            token=os.environ.get('HF_TOKEN'),
        )
        print(f'✅ 完成 ({(time.time()-t0)/60:.1f} min) | {len(ds_k)} 样本')
    except Exception as e:
        print(f'❌ Kinetics 下载失败: {e}')
elif DOWNLOAD_KINETICS and k_dirs:
    print('✅ [3/3] Kinetics-400 已存在，跳过')
else:
    print('⏭️  [3/3] Kinetics-400 跳过（DOWNLOAD_KINETICS=False）')
    print('   如需下载，将本 Cell 顶部 DOWNLOAD_KINETICS 改为 True')

# ──────────────────────────────────────────
# 清理 Arrow 缓存（节省 Drive 空间）
# ──────────────────────────────────────────
arrow_files = glob.glob(f'{HF_DATA_DIR}/**/*.arrow', recursive=True)
if arrow_files:
    for f in arrow_files:
        os.remove(f)
    print(f'\n🧹 已清理 {len(arrow_files)} 个 Arrow 缓存文件（Drive 空间已节省）')
else:
    print('\n🧹 无 Arrow 缓存需要清理')

# ──────────────────────────────────────────
# 汇总
# ──────────────────────────────────────────
total_size = sum(f.stat().st_size for f in Path(HF_DATA_DIR).rglob('*') if f.is_file())
_, _, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'\n📊 HF 数据目录大小: {total_size/1e9:.2f} GB')
print(f'📊 Drive 剩余空间:  {free/1e9:.1f} GB')
print('\n✅ 数据集准备完成！下一步请运行 Cell 7（初始化模型）')

---
## 🤖 Cell 7：初始化模型（验证所有修复生效）

In [ ]:
import torch
import sys
sys.path.insert(0, '/content/NSCA')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

# 使用修复后的 create_cognitive_agent
agent = create_cognitive_agent_fixed(use_llm=False).to(device)

# ── 验证维度修复 ──
print('\n维度验证：')

# HardnessExtractor.audio_encoder 输入维度应为 512
ae_dim = agent.property_layer.hardness_extractor.audio_encoder[0].in_features
print(f'  hardness_extractor.audio_encoder 输入维度: {ae_dim}', 
      '✅' if ae_dim == 512 else f'❌ 期望 512')

# WeightExtractor.proprio_encoder 输入维度应为 256
pe_dim = agent.property_layer.weight_extractor.proprio_encoder[0].in_features
print(f'  weight_extractor.proprio_encoder 输入维度: {pe_dim}',
      '✅' if pe_dim == 256 else f'❌ 期望 256')

# AudioEncoder 输出维度
ae_out = agent.world_model.audio_encoder.out_channels
print(f'  AudioEncoder 输出维度: {ae_out}',
      '✅' if ae_out == 512 else f'❌')

# ── 前向传播测试 ──
print('\n前向传播测试：')
vision  = torch.randn(1, 2, 3, 64, 64).to(device)
audio   = torch.randn(1, 16000).to(device)
proprio = torch.randn(1, 2, 12).to(device)

try:
    with torch.no_grad():
        result = agent.perceive(vision, audio, proprio)
    print(f'  World state shape: {result["world_state"].shape}')
    print(f'  Hardness: {result["properties"].hardness.item():.4f}')
    print(f'  Curiosity: {result["drive_state"].curiosity_level:.4f}')
    print('  ✅ 前向传播成功！')
except Exception as e:
    print(f'  ❌ 前向传播失败: {e}')
    import traceback; traceback.print_exc()

# 参数量统计
total_params = sum(p.numel() for p in agent.parameters())
trainable_params = sum(p.numel() for p in agent.parameters() if p.requires_grad)
print(f'\n模型参数: {total_params/1e6:.2f}M 总计, {trainable_params/1e6:.2f}M 可训练')

---
## 🗣️ Cell 8：Babbling 阶段（Language Grounding Layer 4）

In [ ]:
import torch, sys, json
from pathlib import Path

sys.path.insert(0, '/content/NSCA')
from src.learning.curriculum_babbling import (
    CurriculumBabbling, BabblingConfig, SimulatedBabblingEnvironment
)

BABBLING_STEPS    = 50_000   # Phase 1: 10k + Phase 2: 40k
SAVE_BABBLING_TO  = '/content/drive/MyDrive/datasets/babbling_results.json'

# ── 如果上次 babbling 已完成，直接加载结果 ──
if Path(SAVE_BABBLING_TO).exists():
    with open(SAVE_BABBLING_TO) as f:
        prev = json.load(f)
    if prev.get('total_steps', 0) >= BABBLING_STEPS:
        print(f'✅ 已有完整的 babbling 结果（{prev["total_steps"]} 步），跳过')
        print(f'   Grounded objects: {prev["grounded_objects"]}')
        print(f'   Learnability scores: {len(prev["learnability_scores"])} 个动作')
        babbling_done = True
    else:
        print(f'⚠️  上次 babbling 未完成（{prev["total_steps"]} 步），重新运行')
        babbling_done = False
else:
    babbling_done = False

if not babbling_done:
    print(f'🗣️  开始 Babbling（共 {BABBLING_STEPS:,} 步）...')
    print(f'   Phase 1 (随机探索): 前 10,000 步')
    print(f'   Phase 2 (能力驱动): 后 40,000 步')
    print()

    config = BabblingConfig(
        phase1_steps=10_000,
        phase2_steps=40_000,
        min_action_history=5,
        learnability_window=20,
        temperature=1.0,
        exploration_bonus=1.0,
    )
    babbling = CurriculumBabbling(config)
    env = SimulatedBabblingEnvironment(noise_level=0.05, use_set='A')

    # 获取 grounder（从已初始化的 agent）
    grounder = agent.language.grounder

    # 运行修复版 babbling
    results = run_babbling_phase_fixed(
        babbling=babbling,
        env=env,
        predictor=None,
        grounder=grounder,
        verbose_every=5000,
    )

    # 保存结果到 Drive
    with open(SAVE_BABBLING_TO, 'w') as f:
        json.dump(results, f, indent=2)

    print(f'\n✅ Babbling 完成！')
    print(f'   Total steps:        {results["total_steps"]}')
    print(f'   Grounded objects:   {results["grounded_objects"]}')
    print(f'   Learnability scores ({len(results["learnability_scores"])} 个动作):')
    for action, score in results['learnability_scores'].items():
        status = '✅ 可学' if score > 0.3 else ('⚠️ 低' if score > 0.1 else '❌ 难学')
        print(f'     {action:10s}: {score:.3f}  {status}')
    print(f'\n   结果已保存到: {SAVE_BABBLING_TO}')

---
## 🏋️ Cell 9：Layer 0 训练（World Model）
> 这是最耗时的阶段，约 2-4 小时（A100）

In [ ]:
import os, subprocess, sys
from pathlib import Path

CHECKPOINT_DIR = '/content/drive/MyDrive/nsca_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

LAYER0_CKPT = f'{CHECKPOINT_DIR}/layer0_world_model.pth'

if Path(LAYER0_CKPT).exists():
    print(f'✅ Layer 0 checkpoint 已存在: {LAYER0_CKPT}')
    print('   如需重新训练，删除该文件后重新运行本 Cell')
else:
    print('🏋️  开始 Layer 0 (World Model) 训练...')

    # 找到训练脚本
    train_script = '/content/NSCA/scripts/train_world_model.py'
    if not Path(train_script).exists():
        train_script = '/content/NSCA/train.py'

    cmd = [
        sys.executable, train_script,
        '--config', '/content/NSCA/configs/training_config.yaml',
        '--phase', 'all',
        '--data-dir', '/content/drive/MyDrive/datasets/vis-data',
        '--checkpoint-dir', CHECKPOINT_DIR,
        '--save-every', '5',
    ]

    print(f'命令: {" ".join(cmd)}')
    result = subprocess.run(cmd)

    if result.returncode == 0:
        print(f'✅ Layer 0 训练完成')
    else:
        print(f'⚠️  训练脚本返回非零退出码: {result.returncode}')
        print('请检查日志，可能需要调整参数')

---
## 🧪 Cell 10：Layer 1 训练（Semantic Properties）
> ⚠️ 训练前会自动验证 audio_encoder 维度，防止再次出现 Loss=0 的问题

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import sys, json
sys.path.insert(0, '/content/NSCA')

CHECKPOINT_DIR = '/content/drive/MyDrive/nsca_checkpoints'
LAYER0_CKPT    = f'{CHECKPOINT_DIR}/layer0_world_model.pth'
LAYER1_CKPT    = f'{CHECKPOINT_DIR}/layer1_properties.pth'

device = torch.device('cuda')

if Path(LAYER1_CKPT).exists():
    print(f'✅ Layer 1 checkpoint 已存在: {LAYER1_CKPT}')
else:
    print('🧪 开始 Layer 1 (Semantic Properties) 训练...')

    # 加载 Layer 0 权重（如果存在）
    if Path(LAYER0_CKPT).exists():
        ckpt_result = load_checkpoint_safe(agent.world_model, LAYER0_CKPT, device)
        print(f'  Layer 0 权重加载: {ckpt_result["loaded"]}/{ckpt_result["total"]}')

    # ── 验证维度（防御性检查）──
    ae_in = agent.property_layer.hardness_extractor.audio_encoder[0].in_features
    pe_in = agent.property_layer.weight_extractor.proprio_encoder[0].in_features
    assert ae_in == 512, f'❌ audio_encoder 维度错误: {ae_in}，应为 512！重新运行 Cell 4'
    assert pe_in == 256, f'❌ proprio_encoder 维度错误: {pe_in}，应为 256！重新运行 Cell 4'
    print(f'  ✅ 维度验证通过: audio_enc={ae_in}, proprio_enc={pe_in}')

    # ── 训练 Layer 1 ──
    optimizer = optim.AdamW(agent.property_layer.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

    # 冻结 Layer 0，只训练 Layer 1
    for p in agent.world_model.parameters():
        p.requires_grad = False
    for p in agent.property_layer.parameters():
        p.requires_grad = True

    agent.train()
    EPOCHS = 20
    STEPS_PER_EPOCH = 500

    print(f'  训练 {EPOCHS} epochs × {STEPS_PER_EPOCH} steps...')
    loss_history = []

    # ── 自动保存：每 30 分钟存一次 checkpoint，断线最多丢 30 分钟进度 ──
    import time as _time
    _last_autosave = _time.time()
    AUTO_SAVE_INTERVAL = 1800  # 30 分钟（秒）
    AUTO_SAVE_DIR = '/content/drive/MyDrive/nsca_checkpoints'
    os.makedirs(AUTO_SAVE_DIR, exist_ok=True)

    for epoch in range(EPOCHS):
        epoch_losses = []

        for step in range(STEPS_PER_EPOCH):
            # 生成训练 batch（此处用合成数据；真实训练替换为真实 DataLoader）
            B = 32
            world_state   = torch.randn(B, 256, device=device)
            audio_feat    = torch.randn(B, 512, device=device)   # ✅ 512维
            proprio_feat  = torch.randn(B, 512, device=device)   # ✅ 512维

            # 生成伪标签（来自 babbling 的 sensory feedback）
            # 真实训练中应从 grounder.grounding_table 取
            hardness_target = torch.rand(B, device=device)
            weight_target   = torch.rand(B, device=device)

            optimizer.zero_grad()

            # Forward
            properties, prop_embed = agent.property_layer(
                world_state, audio_feat, proprio_feat
            )

            # Loss（BCELoss 适合 [0,1] 回归）
            loss_h = nn.functional.binary_cross_entropy(
                properties.hardness, hardness_target)
            loss_w = nn.functional.binary_cross_entropy(
                properties.weight, weight_target)
            loss = loss_h + loss_w

            # ✅ FIX 4 健康检查
            validate_layer1_loss(loss.item(), epoch * STEPS_PER_EPOCH + step)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(agent.property_layer.parameters(), 1.0)
            optimizer.step()
            epoch_losses.append(loss.item())

        scheduler.step()
        mean_loss = sum(epoch_losses) / len(epoch_losses)
        loss_history.append(mean_loss)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:3d}/{EPOCHS} | Loss: {mean_loss:.4f} | '
                  f'LR: {scheduler.get_last_lr()[0]:.2e}')

        # ── 每 30 分钟自动保存（防断线丢失进度）──
        if _time.time() - _last_autosave > AUTO_SAVE_INTERVAL:
            _auto_path = f'{AUTO_SAVE_DIR}/layer1_autosave_epoch{epoch+1}.pth'
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': agent.property_layer.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': mean_loss,
                'loss_history': loss_history,
            }, _auto_path)
            print(f'  💾 自动保存 epoch {epoch+1} → {_auto_path}')
            _last_autosave = _time.time()

    # 解冻所有参数
    for p in agent.world_model.parameters():
        p.requires_grad = True

    # 保存
    torch.save({
        'epoch': EPOCHS,
        'model': agent.property_layer.state_dict(),
        'loss_history': loss_history,
        'final_loss': loss_history[-1],
    }, LAYER1_CKPT)

    print(f'\n✅ Layer 1 训练完成！')
    print(f'   初始 Loss: {loss_history[0]:.4f} → 最终 Loss: {loss_history[-1]:.4f}')
    converged = loss_history[-1] < loss_history[0] * 0.8
    print(f'   收敛状态: {"✅ 正常收敛" if converged else "⚠️ 未充分收敛，建议增加 epochs"}')
    print(f'   Checkpoint: {LAYER1_CKPT}')

---
## 🔬 Cell 11：完整模型评估（对标原始结果）

In [ ]:
import torch, json
from pathlib import Path
import sys
sys.path.insert(0, '/content/NSCA')

device = torch.device('cuda')
agent.eval()

print('=' * 60)
print('📊 模型评估报告')
print('=' * 60)

# ── Layer 1 维度验证 ──
ae_in = agent.property_layer.hardness_extractor.audio_encoder[0].in_features
pe_in = agent.property_layer.weight_extractor.proprio_encoder[0].in_features
print(f'\n[Layer 1 维度]')
print(f'  audio_encoder 输入:  {ae_in}  {"✅" if ae_in==512 else "❌"}')
print(f'  proprio_encoder 输入: {pe_in}  {"✅" if pe_in==512 else "❌"}')

# ── Babbling 结果 ──
SAVE_BABBLING_TO = '/content/drive/MyDrive/datasets/babbling_results.json'
print(f'\n[Layer 4 Language Grounding]')
if Path(SAVE_BABBLING_TO).exists():
    with open(SAVE_BABBLING_TO) as f:
        bres = json.load(f)
    print(f'  Total babbling steps:    {bres["total_steps"]}')
    print(f'  Grounded objects:        {bres["grounded_objects"]}  {"✅" if bres["grounded_objects"]>0 else "❌"}')
    n_scores = len(bres['learnability_scores'])
    print(f'  Learnability scores:     {n_scores} 个动作  {"✅" if n_scores>0 else "❌"}')
    if bres['learnability_scores']:
        avg_score = sum(bres['learnability_scores'].values()) / n_scores
        print(f'  平均 learnability:       {avg_score:.3f}')
else:
    print('  ⚠️  未找到 babbling 结果，请先运行 Cell 8')

# ── 前向传播测试（全模态）──
print(f'\n[前向传播测试]')
with torch.no_grad():
    vision  = torch.randn(1, 2, 3, 64, 64).to(device)
    audio   = torch.randn(1, 16000).to(device)
    proprio = torch.randn(1, 2, 12).to(device)

    try:
        result = agent.perceive(vision, audio, proprio)
        print(f'  World state:       {result["world_state"].shape}  ✅')
        h = result['properties'].hardness.item()
        print(f'  Hardness:          {h:.4f}  {"✅" if 0.01 < h < 0.99 else "⚠️ 可能偏置"}')
        print(f'  Weight:            {result["properties"].weight.item():.4f}')
        print(f'  Curiosity:         {result["drive_state"].curiosity_level:.4f}')

        # 语言 grounding 测试
        match_word, conf = agent.language.find_matching_word(
            result['properties'].to_tensor()
        )
        print(f'  Matched concept:   {match_word} (conf={conf:.4f})  {"✅" if match_word != "unknown" else "⚠️ 需更多 babbling"}')
    except Exception as e:
        print(f'  ❌ 失败: {e}')
        import traceback; traceback.print_exc()

# ── 对比修复前后 ──
print(f'\n[修复前后对比]')
print(f'{"指标":<35} {"修复前":>12} {"修复后":>12}')
print('-' * 60)
print(f'{"audio_encoder 输入维度":<35} {"256":>12} {str(ae_in):>12}')
print(f'{"Layer1 loss 坍塌"::<35} {"是":>12} {"已检测":>12}')
n_grounded = bres.get('grounded_objects', 0) if Path(SAVE_BABBLING_TO).exists() else '待运行'
print(f'{"learnability_scores"::<35} {"空 {}":>12} {str(n_scores)+" 个":>12}')
print(f'{"Checkpoint 加载".ljust(35)} {"810/814":>12} {"自动重映射":>12}')

print('\n✅ 评估完成！')

---
## 💾 Cell 12：保存完整 Checkpoint 到 Google Drive

In [ ]:
import torch, os
from datetime import datetime

CHECKPOINT_DIR = '/content/drive/MyDrive/nsca_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
FULL_CKPT = f'{CHECKPOINT_DIR}/nsca_full_{timestamp}.pth'

torch.save({
    'model': agent.state_dict(),
    'config': {
        'audio_dim': 512,
        'proprio_dim': 512,
        'world_state_dim': 256,
    },
    'fixes_applied': [
        'audio_encoder_dim_512',
        'proprio_encoder_dim_512',
        'babbling_grounder_integrated',
        'checkpoint_key_remapping',
        'layer1_loss_validation',
    ],
    'timestamp': timestamp,
}, FULL_CKPT)

size_mb = os.path.getsize(FULL_CKPT) / 1e6
print(f'✅ 完整模型已保存: {FULL_CKPT} ({size_mb:.0f} MB)')
print('\n⚠️  重要提示：每次成功训练后请运行本 Cell！')
print('   Colab 会话结束后 /content 会清空，只有 Drive 的文件是永久的。')

---
## 📋 附录：问题诊断速查表

| 症状 | 原因 | 解决方法 |
|------|------|----------|
| `RuntimeError: size mismatch` in audio_encoder | 未运行 Cell 4 | 重新运行 Cell 4 |
| Layer 1 loss = 0.0000 | targets 方差为 0 或 DataLoader 空 | Cell 10 会自动警告 |
| learnability_scores = {} | Cell 8 未传入 grounder | Cell 8 已修复 |
| Checkpoint 加载 810/814 | 维度不匹配的 2 个 key | Cell 4 修复后维度一致，应 814/814 |
| Colab 断线后代码失效 | /content 被清空 | 从 Cell 3 重新开始，Drive 数据不丢失 |